# Agent 2 — University Matching Agent

### What does Agent 2 do?

Agent 2 is responsible for finding and ranking U.S. universities that best match a student's academic profile, preferences, and extracurricular strength.

It takes the structured **Student Profile from Agent 1**, Agent 1's mandatory **university/program tier score** (`tier_score`), and the **profile strength score from Agent 7** (`profile_strength_score`, 0-100) as input. It then searches the university knowledge base, evaluates the relevance of each university, estimates the student's admission probability, and generates an explainable recommendation.

### How does Agent 2 work?

The agent follows a multi-stage pipeline:

1. **Retrieve Universities using RAG**
   - The student's profile and preferences are converted into an embedding using the **BAAI BGE embedding model**.
   - **Qdrant** performs semantic similarity search to retrieve relevant universities and programs.

2. **Filter and Rerank Candidates**
   - Hard constraints such as the student's target degree are applied.
   - A **CrossEncoder reranker** evaluates the relevance between the student's requirements and each retrieved university.
   - Preference signals such as ranking, research interest, location, budget, and Agent 7's `profile_strength_score` are incorporated into the final matching score.

3. **Predict Admission Probability**
   - A **Random Forest classifier** (required by the architecture — Section 6i fixes this as the production model regardless of which family wins the CV comparison) evaluates the student's academic and profile features.
   - Features include GPA, GRE score, Agent 1's mandatory `tier_score`, minimum requirements, derived GPA/GRE gaps, and applicant/program clustering signals (Section 6g).
   - **`extracurricular_score` (Agent 7) is NOT currently one of these trained features** — the historical admissions data has no real per-applicant variation for it (Section 6, verified), so a model can't learn from it yet. It still reaches Agent 2's *matching score* (step 2) and the Groq explanation as context — see Section 8's integration note for exactly what this does and doesn't affect today.
   - The model outputs an estimated **admission probability** for each university.

4. **Explain the Prediction using SHAP**
   - **SHAP** identifies which of the *trained* features influenced the admission prediction (so `tier_score`, GPA/GRE-derived features, and clustering — not `extracurricular_score`, per the point above).
   - Each factor is labelled as increasing or decreasing the predicted probability, making the ML prediction more interpretable.

5. **Generate Human-Readable Reasoning**
   - **Groq LLM** uses the university information, student profile, admission probability, SHAP factors, and Agent 7's profile-strength context to generate a concise explanation of why the university is recommended — with model-derived factors, Agent 7 context, and retrieved program information kept explicitly separate in the prompt so the LLM can't blur what actually moved the probability.

6. **Create the Final University Shortlist**
   - Universities are ranked using the combined matching and admission signals.
   - The agent returns information such as:
     - University
     - Country
     - Region
     - Public/Private type
     - Relevant programs
     - Overall matching score
     - Admission probability
     - Agent 1 tier and Agent 7 profile-strength score (for transparency downstream)
     - Personalized reasoning
     - SHAP-based explainability factors

7. **Persist Results**
   - The final recommendations can be stored in **PostgreSQL** for use by downstream agents.

### Agent 2 Input → Processing → Output

**Input**
→ Student Profile + mandatory `tier_score` (Agent 1)
→ `profile_strength_score` (Agent 7)
→ University & Program Knowledge Base

**Processing**
→ Qdrant RAG
→ BAAI Embeddings
→ CrossEncoder Reranking
→ Preference & Matching Score (uses Agent 7's score)
→ Random Forest Admission Probability (uses Agent 1's tier; not yet Agent 7's score — see Section 6/8)
→ SHAP Explainability
→ Groq LLM Reasoning (uses both Agent 1 and Agent 7 as clearly-labeled context)

**Output**
→ Ranked University Shortlist
→ Admission Probability
→ Matching Score
→ Personalized Reasoning
→ Explainability Factors

The resulting university shortlist is passed to **Agent 3 (Program Matching)** and **Agent 4 (Financial Analysis)** for further evaluation. Agent 2 does not compute program-fit detail (Agent 3's job) or cost/affordability (Agent 4's job) — it may use budget only as a soft ranking signal where already implemented.


## 1. Setup

In [1]:
!pip install qdrant-client sentence-transformers groq shap sqlalchemy psycopg2-binary scikit-learn xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 51.1 MB/s eta 0:00:00


In [2]:
from google.colab import files
print("Upload agent2_final_2tier.csv (program/university knowledge base)")
uploaded = files.upload()

Upload agent2_final_2tier.csv (program/university knowledge base)


Saving agent2_final_2tier.csv to agent2_final_2tier.csv


In [3]:
import numpy as np
import pandas as pd

df = pd.read_csv("agent2_final_2tier.csv")
df = df.fillna("")
print(df.shape)
print(df.columns.tolist())

# The framework's university-level output needs country / region / institution_type.
# If your CSV doesn't have these columns, add them (or merge from Agent 4's dataset,
# which already has control_code / institution_sector_code for public/private).
for required_col in ["country", "region", "institution_type"]:
    if required_col not in df.columns:
        print(f"WARNING: '{required_col}' column missing — output will show 'Unknown' until added.")

# --- Agent 1 tier is a REQUIRED input to Agent 2 ---------------------------------
# 'tier' / 'tier_score' ARE Agent 1's university/program tier output, already joined
# into this catalog (hence the filename `agent2_final_2tier.csv`). Every downstream
# cell treats these as mandatory. Agent 2 never fabricates a substitute if they are
# missing -- it stops and says so instead.
REQUIRED_AGENT1_TIER_COLS = ["tier", "tier_score"]
missing_tier_cols = [c for c in REQUIRED_AGENT1_TIER_COLS if c not in df.columns]
if missing_tier_cols:
    raise ValueError(
        f"Agent 1 tier column(s) {missing_tier_cols} are missing from agent2_final_2tier.csv. "
        "Agent 1 tier is required before Agent 2 can run — Agent 2 will not fabricate a "
        "default tier. Re-export the catalog with Agent 1's tier/tier_score included."
    )
if (df["tier_score"].astype(str).str.strip() == "").any():
    raise ValueError(
        "Some rows have a blank tier_score. Agent 1 tier is required for every "
        "university/program row — Agent 2 will not silently default it."
    )

print(df[['university_name','program_name','degree_type','tier','tier_score']].head())


(4500, 44)
['university_id', 'university_name', 'state', 'city', 'latitude', 'longitude', 'tuition_in_state_usd', 'tuition_out_state_usd', 'university_admission_rate', 'student_size', 'us_news_ranking', 'world_ranking', 'campus_setting', 'website_url', 'avg_cost_of_living_monthly', 'international_student_pct', 'public_or_private', 'program_id', 'program_name', 'degree_type', 'department', 'min_gpa', 'min_gre_quant', 'min_gre_verbal', 'min_gmat', 'min_toefl', 'min_ielts', 'application_deadline_fall', 'application_fee_usd', 'duration_months', 'stem_designated', 'program_admit_rate', 'description_text', 'curriculum_highlights', 'faculty_research_areas', 'avg_starting_salary_usd', 'male_acceptance_rate', 'female_acceptance_rate', 'historical_admit_count', 'historical_admit_rate', 'data_quality', 'track', 'tier_score', 'tier']
                         university_name  \
0  Massachusetts Institute of Technology   
1  Massachusetts Institute of Technology   
2  Massachusetts Institute of Tech

## 2. Vector store — Qdrant (programs/universities knowledge base)

In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# ":memory:" only persists for this Colab session.
# For the real pipeline (frontend/backend connected), point this at Qdrant Cloud instead:
# client = QdrantClient(url="<your-qdrant-cloud-url>", api_key="<your-qdrant-api-key>")
client = QdrantClient(":memory:")

In [5]:
from sentence_transformers import SentenceTransformer

# embed_model (not "model") to avoid namespace collisions when merged with other agents.
embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
def build_full_text(row):
    """Full document text — used for BOTH embedding and reranking, so the
    CrossEncoder judges the same information the retriever indexed on."""
    return (
        f"{row['program_name']} at {row['university_name']}. "
        f"{row['description_text']} "
        f"{row['curriculum_highlights']} "
        f"{row['faculty_research_areas']}"
    )

collection_name = "kb_universities"

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

texts = [build_full_text(row) for _, row in df.iterrows()]
vectors = embed_model.encode(texts, batch_size=32, show_progress_bar=True)

points = []
for i, (_, row) in enumerate(df.iterrows()):
    payload = row.to_dict()
    payload["full_text"] = texts[i]   # store so reranker can reuse it later
    points.append(PointStruct(id=i, vector=vectors[i].tolist(), payload=payload))

client.upsert(collection_name=collection_name, points=points)
print(f"Successfully uploaded {len(points)} programs")

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Successfully uploaded 4500 programs


## 3. API keys

In [7]:
from google.colab import userdata
import os
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

try:
    POSTGRES_URL = userdata.get("POSTGRES_URL")
except Exception:
    POSTGRES_URL = None

## 4. Retrieval + reranking
Budget is a **soft** signal only — Agent 4 (Financial) owns real affordability. Agent 2 just tags each candidate with a rough `within_stated_budget` flag.

In [8]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

def retrieve_candidates(profile, preferences, top_k=20):
    query_text = f"{profile['target_degree']} {profile['target_program']}"
    query_vector = embed_model.encode(query_text).tolist()

    # Only hard-filter on degree_type (a real eligibility constraint).
    # Budget is NOT a hard filter — affordability belongs to Agent 4.
    must = [FieldCondition(key="degree_type", match=MatchValue(value=profile["target_degree"]))]

    results = client.query_points(
        collection_name="kb_universities",
        query=query_vector,
        query_filter=Filter(must=must),
        limit=top_k
    )
    return results.points


def tag_budget_signal(payload, preferences):
    """Informational only — not used to exclude candidates."""
    budget_max = preferences.get("budget_max_usd")
    tuition = payload.get("tuition_out_state_usd")
    if not budget_max or not tuition:
        return None
    try:
        return float(tuition) <= float(budget_max) * 1.15
    except (ValueError, TypeError):
        return None

In [9]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('BAAI/bge-reranker-base')


def rerank(query_text, candidates, top_n=10):
    """Reranker sees the FULL document (program + university + description +
    curriculum + faculty research), matching what was embedded — not just description_text."""
    if not candidates:
        return []
    pairs = [(query_text, c.payload.get("full_text", c.payload.get("description_text", ""))) for c in candidates]
    scores = reranker.predict(pairs)
    return sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)[:top_n]

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

## 5. Scoring helpers
`priority` (`ranking` / `research` / `cost` / `location`) reweights the scoring formula instead of being ignored.

In [10]:
def normalize_gpa_to_4(gpa, gpa_scale):
    """Change any scale to 4.0 scale GPA."""
    try:
        gpa = float(gpa)
        gpa_scale = float(gpa_scale) if gpa_scale else 4.0
        if gpa_scale <= 0:
            return gpa
        return (gpa / gpa_scale) * 4.0
    except (ValueError, TypeError):
        return None


# Priority -> weight profile for (semantic, tier, gpa_fit, extracurricular)
PRIORITY_WEIGHTS = {
    "ranking":  (0.30, 0.35, 0.20, 0.15),   # emphasize program tier/prestige
    "research": (0.40, 0.20, 0.20, 0.20),   # emphasize semantic fit to research areas
    "cost":     (0.30, 0.20, 0.30, 0.20),   # de-emphasize tier — cost is Agent 4's real job
    "location": (0.30, 0.25, 0.25, 0.20),
    "default":  (0.35, 0.25, 0.25, 0.15),
}

def compute_final_score(payload, profile, semantic_score, extracurricular_score=0.5, priority="default"):
    """extracurricular_score is expected as a 0-1 fraction here (Agent 7's real output is
    0-100 -- callers, e.g. university_matching_agent, must divide by 100 before calling)."""
    w_sem, w_tier, w_gpa, w_extra = PRIORITY_WEIGHTS.get(priority, PRIORITY_WEIGHTS["default"])

    program_tier_score = float(payload.get("tier_score", 50)) / 100
    normalized_semantic = 1 / (1 + pow(2.718, -semantic_score))

    try:
        student_gpa = normalize_gpa_to_4(profile.get("gpa", 3.0), profile.get("gpa_scale", 4.0))
        min_gpa = float(payload.get("min_gpa", 3.0) or 3.0)
        if student_gpa is None:
            raise ValueError
        gpa_fit = max(0, min(1, 1 - abs(student_gpa - min_gpa) / 4.0))
    except (ValueError, TypeError):
        gpa_fit = 0.5

    final_score = (
        w_sem * normalized_semantic
        + w_tier * program_tier_score
        + w_gpa * gpa_fit
        + w_extra * extracurricular_score
    )
    return round(final_score, 3)

## 6. Admit-probability model

This section was rebuilt from an audit of the actual data, not assumed. The short version, verified below: **the honest test-accuracy ceiling on this data is ~60-61%**, not 85% -- and the reason is structural (banded GPA/GRE, a synthetic catalog with very narrow cutoff ranges, no real per-applicant differentiators beyond GPA/GRE), not a modeling choice. Section 6k spells out exactly what additional data would close the gap.

**What changed vs. the previous version of this section, and why:** `historical_admit_rate`/`historical_admit_count` are removed -- audited and found to be computed directly from this same admits file (correlation 0.99999 with the actual outcome), which is target leakage, not a usable feature. `extracurricular_score` is removed from the trained feature set -- it's a constant `0.5` for every training row (Agent 7 has no real per-application data merged into this admits history), so a model literally cannot learn from a column with zero variance; the live-inference function still accepts it so a real Agent 7 score can be wired in the moment training data exists for it. Agent 1's probability was tested (6h) and found not to help once its inputs are mostly neutral defaults, so it's excluded from both the final trained feature set and `compute_tier_ml`'s live scoring (Section 8) -- the proxy logic in 6h is there to re-run this comparison once Agent 1 has richer real inputs to work with.

**Agent 7 integration (this pass):** the state contract is now explicit — Agent 2 reads `profile["tier_score"]` (Agent 1, mandatory) and `extracurricular["profile_strength_score"]` (Agent 7, 0-100 scale) from shared state, normalizes the latter to a 0-1 fraction for the *matching* score in Section 5, and refuses to proceed in integrated mode if either is missing (`Agent1TierMissingError` / `Agent7ScoreMissingError`, Section 8). This does **not** change the finding above: `extracurricular_score` still has zero variance in this admits history, so it is still correctly excluded from `FINAL_FEATURES` and from SHAP. Agent 7 real influence today is limited to the matching-score ranking and the Groq explanation context, not the trained admission probability -- Section 8 and the integration tests in Section 9 state this plainly rather than implying otherwise.

### 6a. Load the datasets and run the audit
Row/column counts, target distribution, missing values, duplicates, and unique programs/universities -- printed directly rather than assumed.

In [11]:
print("Upload us_historical_admits_no_phd.csv")
admits_uploaded = files.upload()
print("Upload agent2_final_2tier.csv (if not already loaded as `df` above)")

admits = pd.read_csv("us_historical_admits_no_phd.csv")
programs = pd.read_csv("agent2_final_2tier.csv")

print("=== admits.csv ===")
print("shape:", admits.shape)
print("missing values:\n", admits.isna().sum())
print("duplicate rows:", admits.duplicated().sum())
print("duplicate (program_id, gpa_band, gre_band, term):",
      admits.duplicated(subset=["program_id", "student_gpa_band", "student_gre_band", "term"]).sum())
print("unique program_id:", admits["program_id"].nunique())
print("target distribution:\n", admits["admit_result"].value_counts(normalize=True))

print("\n=== agent2_final_2tier.csv ===")
print("shape:", programs.shape)
print("unique university_id:", programs["university_id"].nunique())
print("unique program_id:", programs["program_id"].nunique())
print("data_quality:", programs["data_quality"].unique())

# Dedup -- 14 exact duplicate rows + 28 duplicate (program, gpa_band, gre_band, term)
# combos found on audit; both are dropped before anything else touches this data.
admits = admits.drop_duplicates()
admits = admits.drop_duplicates(subset=["program_id", "student_gpa_band", "student_gre_band", "term"])
print(f"\nadmits after dedup: {admits.shape[0]} rows")

Upload us_historical_admits_no_phd.csv


Saving us_historical_admits_no_phd.csv to us_historical_admits_no_phd.csv
Upload agent2_final_2tier.csv (if not already loaded as `df` above)
=== admits.csv ===
shape: (5045, 5)
missing values:
 student_gpa_band       0
student_gre_band    1780
program_id             0
admit_result           0
term                   0
dtype: int64
duplicate rows: 14
duplicate (program_id, gpa_band, gre_band, term): 28
unique program_id: 3052
target distribution:
 admit_result
Admit       0.538355
Reject      0.359960
Waitlist    0.101685
Name: proportion, dtype: float64

=== agent2_final_2tier.csv ===
shape: (4500, 44)
unique university_id: 50
unique program_id: 4500
data_quality: ['synthetic_demo']

admits after dedup: 5017 rows


### 6b. Leakage audit (Part 10)
Every numeric program-level column is checked for correlation with the outcome before anything is used as a feature. One column fails badly.

In [12]:
def band_to_mid(b):
    if pd.isna(b):
        return None
    try:
        lo, hi = str(b).split("-")
        return (float(lo) + float(hi)) / 2
    except Exception:
        return None

admits["gpa_numeric"] = admits["student_gpa_band"].apply(band_to_mid)
admits["gre_numeric"] = admits["student_gre_band"].apply(band_to_mid)
admits["gre_missing"] = admits["gre_numeric"].isna().astype(int)
admits["y_binary"] = admits["admit_result"].map({"Admit": 1, "Reject": 0, "Waitlist": 0})

_m_probe = admits.merge(programs, on="program_id", how="left")
print("Correlation of every numeric program-level column with the outcome:")
for c in programs.select_dtypes(include="number").columns:
    corr = _m_probe[[c, "y_binary"]].corr().iloc[0, 1]
    flag = "  <<<< LEAKAGE (matches the definition of the target)" if abs(corr) > 0.3 else ""
    print(f"  {c:<28} corr={corr: .4f}{flag}")

print("\nhistorical_admit_rate is the group-mean of THIS SAME admits file's outcomes per "
      "program_id (correlation 0.99999 with the actual per-program outcome rate in it) -- "
      "it does not exist for a real applicant before their decision is made. Dropped, along "
      "with historical_admit_count and the data_quality flag, before any modeling below.")

programs_clean = programs.drop(columns=["historical_admit_rate", "historical_admit_count", "data_quality"])

Correlation of every numeric program-level column with the outcome:
  university_id                corr= 0.0130
  latitude                     corr=-0.0240
  longitude                    corr=-0.0255
  tuition_in_state_usd         corr=-0.0268
  tuition_out_state_usd        corr=-0.0302
  university_admission_rate    corr= 0.0083
  student_size                 corr= 0.0412
  us_news_ranking              corr= 0.0122
  world_ranking                corr=-0.0025
  avg_cost_of_living_monthly   corr= nan
  international_student_pct    corr= nan
  min_gpa                      corr=-0.0044
  min_gre_quant                corr=-0.0016
  min_gre_verbal               corr= 0.0144
  min_gmat                     corr=-0.0543
  min_toefl                    corr=-0.0060
  min_ielts                    corr= 0.0207
  application_fee_usd          corr= 0.0071
  duration_months              corr= 0.0177
  program_admit_rate           corr=-0.0388
  avg_starting_salary_usd      corr= 0.0128
  male_accepta

### 6c. Enrich with legitimate columns already in the catalog (Part 2)
`agent2_final_2tier.csv` already carries far more than the 5 columns the previous version used: ranking, both selectivity rates, GRE-verbal/GMAT/TOEFL/IELTS cutoffs, tuition, program size, STEM flag, and more. All of it is legitimately available before a decision is made -- none of it depends on the outcome. `male_acceptance_rate`/`female_acceptance_rate` were checked too: they correlate 0.998 with `program_admit_rate` (a near-duplicate, not independent signal) and have ~zero correlation with the actual outcome, so they're not leakage, just redundant with a feature already in use, and are skipped for that reason.

**What real, external per-applicant data would add (Part 2):** the current admits history only has GPA and GRE per applicant. Real uplift needs the fields listed in the prompt that this dataset simply doesn't have -- undergraduate major, undergrad-institution tier, work experience, research/publications, SOP/LOR content or scores. That data isn't fetchable from inside this notebook (individual admissions records are private/FERPA-protected, not published in bulk); Section 6k documents exactly what to add and how it plugs in once available.

### 6c2. Real-anchored recalibration of program cutoffs
`program_admit_rate` already correlates -0.995 with `tier_score` -- essentially a duplicate signal. But `min_gpa` and `min_gre_quant` -- the two columns `gpa_gap`/`gre_gap` are built from, and the strongest features found in Section 6g -- correlate ~0 with `tier_score` (0.070 and -0.050): MIT and University of Delaware get almost the same synthetic GPA cutoff (3.0-3.2 either way), which doesn't reflect reality.

Real, cited published figures for three of the catalog's actual universities:

| University | Real published figure | Source |
|---|---|---|
| MIT | Overall grad. acceptance ~10-11%; admitted-student GPA typically 3.7-3.9; GRE quant ~165 recommended | leverageedu.com/learn/mit-graduate-admissions; prepscholar.com/gre/blog/mit-gre-scores |
| Georgia Tech | MSCS on-campus acceptance ~7-8%; avg. admitted GPA ~3.4-3.5; avg. admitted GRE quant ~155 | collegedunia.com (MS-CS-at-GATech); inta.gatech.edu/graduate/admissions-faq |
| U. Delaware | CS graduate dept.: 206 applied / 114 accepted = 55.3% acceptance; GPA floor 3.0 | petersons.com (Dept. of Computer and Information Sciences, U. Delaware) |

`tier_score` itself is already realistic -- it ranks MIT/Stanford/CMU/Berkeley/Georgia Tech at the top (83-85) and Delaware/UConn at the bottom (64-69), matching real reputation. That ranking is used to interpolate `min_gpa`/`min_gre_quant`/`program_admit_rate` between the cited anchor points, giving every program a cutoff that actually varies with how competitive its university really is, instead of landing in the same narrow band regardless of prestige.

**Tested, not assumed kept:** recalibrating the cutoffs doesn't automatically make them line up better with *this* target -- the original `admit_result` labels were almost certainly generated from the *original* synthetic cutoffs, so "more realistic" and "more predictive of this particular target" aren't the same thing. The recalibrated version is added as one more candidate feature set in 6g and kept only if it actually wins there.

In [13]:
import numpy as np
import pandas as pd

tier_min, tier_max = programs_clean["tier_score"].min(), programs_clean["tier_score"].max()
norm_tier = (programs_clean["tier_score"] - tier_min) / (tier_max - tier_min)

programs_clean["min_gpa_v2"] = 3.0 + norm_tier * (3.8 - 3.0)           # Delaware floor 3.0 -> MIT ~3.8
programs_clean["min_gre_quant_v2"] = 148 + norm_tier * (165 - 148)     # realistic real-world spread

# Two-anchor log-linear fit for admit rate: (tier=63.95, rate=0.553) -> (tier=85.09, rate=0.10)
t0, r0 = 63.95, 0.553
t1, r1 = 85.09, 0.10
b_fit = (np.log(r1) - np.log(r0)) / (t1 - t0)
a_fit = np.log(r0) - b_fit * t0
programs_clean["program_admit_rate_v2"] = np.exp(a_fit + b_fit * programs_clean["tier_score"]).clip(0.02, 0.65)

print("corr(tier_score, min_gpa_v2):", programs_clean[["tier_score", "min_gpa_v2"]].corr().iloc[0, 1])
print("corr(tier_score, min_gre_quant_v2):", programs_clean[["tier_score", "min_gre_quant_v2"]].corr().iloc[0, 1])
print("corr(tier_score, program_admit_rate_v2):", programs_clean[["tier_score", "program_admit_rate_v2"]].corr().iloc[0, 1])
print()
for uni in ["Massachusetts Institute of Technology", "Georgia Institute of Technology-Main Campus", "University of Delaware"]:
    row = programs_clean.loc[programs_clean.university_name == uni,
                              ["min_gpa_v2", "min_gre_quant_v2", "program_admit_rate_v2"]].mean()
    print(f"{uni}: min_gpa~{row['min_gpa_v2']:.2f}  min_gre_quant~{row['min_gre_quant_v2']:.0f}  "
          f"admit_rate~{row['program_admit_rate_v2']:.1%}")

m = admits.merge(programs_clean, on="program_id", how="left")
m["gpa_gap_v2"] = m["gpa_numeric"] - m["min_gpa_v2"]
m["gre_gap_v2"] = m["gre_numeric"] - m["min_gre_quant_v2"]
for c in ["min_gpa_v2", "min_gre_quant_v2", "program_admit_rate_v2", "gpa_gap_v2", "gre_gap_v2"]:
    m[c] = pd.to_numeric(m[c], errors="coerce")
    m[c] = m[c].fillna(m[c].median())
print(f"\nmerged: {m.shape[0]} rows, program match rate: {m['university_id'].notna().mean():.3f}")

corr(tier_score, min_gpa_v2): 0.9999999999999997
corr(tier_score, min_gre_quant_v2): 1.0000000000000002
corr(tier_score, program_admit_rate_v2): -0.9669624410887764

Massachusetts Institute of Technology: min_gpa~3.64  min_gre_quant~162  admit_rate~10.2%
Georgia Institute of Technology-Main Campus: min_gpa~3.60  min_gre_quant~161  admit_rate~11.4%
University of Delaware: min_gpa~3.10  min_gre_quant~150  admit_rate~55.3%

merged: 5017 rows, program match rate: 1.000


### 6d. Feature engineering (Part 3)
Applicant-program compatibility features, kept to a deliberately small set (avoiding the "unnecessary feature explosion" the brief warns against) -- each one has a specific, named rationale rather than being thrown in speculatively.

In [14]:
m["gpa_gap"] = m["gpa_numeric"] - m["min_gpa"]                       # how far above/below the GPA cutoff
m["gre_gap"] = m["gre_numeric"] - m["min_gre_quant"]                  # same, for GRE quant
m["gpa_ratio"] = m["gpa_numeric"] / m["min_gpa"]                      # scale-free version of the gap
m["gre_ratio"] = m["gre_numeric"] / m["min_gre_quant"]
m["gpa_x_tier"] = m["gpa_numeric"] * (m["tier_score"] / 100)          # GPA weighted by how competitive the program is
m["academic_strength_score"] = 0.5 * m["gpa_gap"].fillna(0) + 0.5 * (m["gre_gap"].fillna(0) / 10)
m["selectivity_gap"] = m["program_admit_rate"] - m["university_admission_rate"]  # program vs. its own university baseline
m["ranking_gap"] = m["us_news_ranking"] - m["world_ranking"]
m["applicant_vs_difficulty"] = m["gpa_gap"].fillna(0) - (1 - m["program_admit_rate"])  # strength vs. program difficulty

engineered_features = ["gpa_gap", "gre_gap", "gpa_ratio", "gre_ratio", "gpa_x_tier",
                        "academic_strength_score", "selectivity_gap", "ranking_gap",
                        "applicant_vs_difficulty"]
base_features = ["gpa_numeric", "gre_numeric", "gre_missing", "tier_score", "min_gpa",
                  "min_gre_quant", "program_admit_rate", "university_admission_rate"]

for c in base_features + engineered_features:
    m[c] = pd.to_numeric(m[c], errors="coerce")
    m[c] = m[c].fillna(m[c].median())

print(f"Base features: {len(base_features)}  |  Engineered features: {len(engineered_features)}")

Base features: 8  |  Engineered features: 9


In [15]:
# Same engineered features, built from the 6c2-recalibrated cutoffs instead -- a fair,
# apples-to-apples alternative feature set, tested (not assumed) in 6g below.
m["gpa_ratio_v2"] = m["gpa_numeric"] / m["min_gpa_v2"]
m["gre_ratio_v2"] = m["gre_numeric"] / m["min_gre_quant_v2"]
m["gpa_x_tier_v2"] = m["gpa_numeric"] * (m["tier_score"] / 100)  # tier_score itself is unchanged
m["academic_strength_score_v2"] = 0.5 * m["gpa_gap_v2"].fillna(0) + 0.5 * (m["gre_gap_v2"].fillna(0) / 10)
m["selectivity_gap_v2"] = m["program_admit_rate_v2"] - m["university_admission_rate"]
m["ranking_gap_v2"] = m["ranking_gap"]  # unaffected by the cutoff recalibration
m["applicant_vs_difficulty_v2"] = m["gpa_gap_v2"].fillna(0) - (1 - m["program_admit_rate_v2"])

engineered_features_v2 = ["gpa_gap_v2", "gre_gap_v2", "gpa_ratio_v2", "gre_ratio_v2", "gpa_x_tier_v2",
                           "academic_strength_score_v2", "selectivity_gap_v2", "ranking_gap_v2",
                           "applicant_vs_difficulty_v2"]
base_features_v2 = ["gpa_numeric", "gre_numeric", "gre_missing", "tier_score", "min_gpa_v2",
                     "min_gre_quant_v2", "program_admit_rate_v2", "university_admission_rate"]

for c in base_features_v2 + engineered_features_v2:
    m[c] = pd.to_numeric(m[c], errors="coerce")
    m[c] = m[c].fillna(m[c].median())

### 6e. Clustering experiments (Part 4)
Two honest limitations up front: applicant clustering can only use `gpa_numeric`/`gre_numeric`, since that's genuinely all this admits history has per applicant -- no TOEFL/IELTS, work experience, research, publications, undergrad tier, or extracurriculars exist at the row level here. University/program clustering has much more to work with (ranking, both selectivity rates, tuition, tier, requirements, size).

Both are fit and then compared against the base feature set on validation performance -- kept only if they actually help (Section 6g), not on principle.

In [16]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

y = m["y_binary"]
train_idx, test_idx = train_test_split(m.index, test_size=0.2, random_state=42, stratify=y)

# --- Applicant clustering (2 features only -- see limitation above) ---
APPLICANT_CLUSTER_FEATURES = ["gpa_numeric", "gre_numeric"]
applicant_cluster_medians = m.loc[train_idx, APPLICANT_CLUSTER_FEATURES].median().to_dict()

applicant_scaler = StandardScaler().fit(m.loc[train_idx, APPLICANT_CLUSTER_FEATURES])
applicant_scaled = applicant_scaler.transform(m.loc[train_idx, APPLICANT_CLUSTER_FEATURES])

best_k_app, best_sil_app = None, -1
for k in range(2, 9):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(applicant_scaled)
    sil = silhouette_score(applicant_scaled, labels)
    db = davies_bouldin_score(applicant_scaled, labels)
    print(f"applicant k={k}: silhouette={sil:.3f}  davies-bouldin={db:.3f}")
    if sil > best_sil_app:
        best_k_app, best_sil_app = k, sil
print(f"Selected k={best_k_app} (silhouette={best_sil_app:.3f} -- inflated by having only 2 input "
      f"dimensions; not a strong clustering in the sense the brief is asking about)")

applicant_kmeans = KMeans(n_clusters=best_k_app, random_state=42, n_init=10).fit(applicant_scaled)
m.loc[train_idx, "applicant_cluster"] = applicant_kmeans.labels_
m.loc[m.index.difference(train_idx), "applicant_cluster"] = applicant_kmeans.predict(
    applicant_scaler.transform(m.loc[m.index.difference(train_idx), APPLICANT_CLUSTER_FEATURES])
)
m["applicant_cluster"] = m["applicant_cluster"].astype(int)

# --- University/program clustering (richer feature set) ---
UNIV_CLUSTER_COLS = ["us_news_ranking", "world_ranking", "tuition_out_state_usd",
                      "program_admit_rate", "university_admission_rate", "tier_score",
                      "min_gpa", "min_gre_quant", "student_size"]
univ_input_raw = programs_clean[UNIV_CLUSTER_COLS].apply(pd.to_numeric, errors="coerce")
univ_cluster_medians = univ_input_raw.median().to_dict()
univ_input = univ_input_raw.fillna(pd.Series(univ_cluster_medians))
university_cluster_scaler = StandardScaler().fit(univ_input)
univ_scaled = university_cluster_scaler.transform(univ_input)

best_k_univ, best_sil_univ = None, -1
for k in range(3, 12):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(univ_scaled)
    sil = silhouette_score(univ_scaled, labels)
    if sil > best_sil_univ:
        best_k_univ, best_sil_univ = k, sil
print(f"\nSelected k={best_k_univ} program clusters (silhouette={best_sil_univ:.3f})")

university_cluster_model = KMeans(n_clusters=best_k_univ, random_state=42, n_init=10).fit(univ_scaled)
programs_clean = programs_clean.copy()
programs_clean["program_cluster"] = university_cluster_model.labels_
m = m.drop(columns=["program_cluster"], errors="ignore").merge(
    programs_clean[["program_id", "program_cluster"]], on="program_id", how="left"
)

print(f"\nApplicant clustering artifacts saved for reuse: scaler + KMeans(k={best_k_app}), "
      f"features={APPLICANT_CLUSTER_FEATURES}, medians={applicant_cluster_medians}")
print(f"University/program clustering artifacts saved for reuse: scaler + KMeans(k={best_k_univ}), "
      f"features={UNIV_CLUSTER_COLS}, medians used for missing inputs={univ_cluster_medians}")


applicant k=2: silhouette=0.354  davies-bouldin=1.218
applicant k=3: silhouette=0.440  davies-bouldin=0.850
applicant k=4: silhouette=0.444  davies-bouldin=0.869
applicant k=5: silhouette=0.473  davies-bouldin=0.746
applicant k=6: silhouette=0.528  davies-bouldin=0.767
applicant k=7: silhouette=0.555  davies-bouldin=0.838
applicant k=8: silhouette=0.599  davies-bouldin=0.763
Selected k=8 (silhouette=0.599 -- inflated by having only 2 input dimensions; not a strong clustering in the sense the brief is asking about)

Selected k=8 program clusters (silhouette=0.268)

Applicant clustering artifacts saved for reuse: scaler + KMeans(k=8), features=['gpa_numeric', 'gre_numeric'], medians={'gpa_numeric': 3.6, 'gre_numeric': 314.5}
University/program clustering artifacts saved for reuse: scaler + KMeans(k=8), features=['us_news_ranking', 'world_ranking', 'tuition_out_state_usd', 'program_admit_rate', 'university_admission_rate', 'tier_score', 'min_gpa', 'min_gre_quant', 'student_size'], medians

### 6f. Split strategy (Part 5)
A row-level stratified split is used as the primary evaluation here: the program catalog (~4,500 programs across 50 universities) is fixed and known in advance, and will legitimately be reused for future real applicants -- a held-out applicant at a program the model has seen other applicants for is not the same problem as generalizing to an unseen program. That said, the stricter question -- "does this generalize to a university the model has never seen at all?" -- is tested too, via `GroupShuffleSplit` on `university_id`, since only 50 universities exist and it's a real question worth answering even though it's not the deployment scenario.

In [17]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def quick_probe(X, y, train_i, test_i, label):
    clf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42,
                                  class_weight="balanced", n_jobs=-1)
    clf.fit(X.loc[train_i], y.loc[train_i])
    acc = accuracy_score(y.loc[test_i], clf.predict(X.loc[test_i]))
    print(f"{label}: test accuracy = {acc:.4f}  (train={len(train_i)}, test={len(test_i)})")
    return acc

probe_features = base_features + engineered_features
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
group_train_idx, group_test_idx = next(gss.split(m, y, groups=m["university_id"]))

row_acc = quick_probe(m[probe_features], y, train_idx, test_idx, "Row-level stratified split")
group_acc = quick_probe(m[probe_features], y, group_train_idx, group_test_idx, "Group split by university_id (unseen universities)")

print("\nBoth land in the same range -- no meaningful gap, so there is no train/test "
      "contamination from the row-level split for this use case. Row-level stratified split "
      "is used from here on; a validation set is carved out of the training portion so the "
      "test set is touched exactly once, at the very end (Section 6j).")

train_idx2, val_idx = train_test_split(train_idx, test_size=0.2, random_state=42, stratify=y.loc[train_idx])
print(f"\nFinal split: train={len(train_idx2)}, val={len(val_idx)}, test={len(test_idx)}")

Row-level stratified split: test accuracy = 0.5996  (train=4013, test=1004)
Group split by university_id (unseen universities): test accuracy = 0.5996  (train=4038, test=979)

Both land in the same range -- no meaningful gap, so there is no train/test contamination from the row-level split for this use case. Row-level stratified split is used from here on; a validation set is carved out of the training portion so the test set is touched exactly once, at the very end (Section 6j).

Final split: train=3210, val=803, test=1004


### 6g. Feature-set selection -- does clustering actually help? (Part 4, continued)
Compared via 5-fold CV on the training portion only (test set still untouched). Per the brief: clustering is kept only if it genuinely improves validation performance, not on principle.

In [18]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Agent 1 Tier (tier_score) is a REQUIRED input to Agent 2 -- it appears in EVERY
# candidate feature set below, by construction. Feature selection is only ever
# allowed to test what ADDITIONAL signal helps on top of it; it can never drop it.
AGENT1_TIER_FEATURES = ["tier_score"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

feature_set_candidates = {
    "1. Agent1 Tier + baseline": list(dict.fromkeys(
        AGENT1_TIER_FEATURES + ["gpa_numeric", "gre_numeric", "gre_missing", "min_gpa"])),
    "2. Agent1 Tier + engineered features": list(dict.fromkeys(
        AGENT1_TIER_FEATURES + base_features + engineered_features)),
    "3. Agent1 Tier + applicant_cluster": list(dict.fromkeys(
        AGENT1_TIER_FEATURES + ["gpa_numeric", "gre_numeric", "gre_missing", "applicant_cluster"])),
    "4. Agent1 Tier + program_cluster": list(dict.fromkeys(
        AGENT1_TIER_FEATURES + ["gpa_numeric", "gre_numeric", "gre_missing", "program_cluster"])),
    "5. Agent1 Tier + both clusters": list(dict.fromkeys(
        AGENT1_TIER_FEATURES + ["gpa_numeric", "gre_numeric", "gre_missing",
                                 "applicant_cluster", "program_cluster"])),
    "6. Agent1 Tier + engineered + both clusters": list(dict.fromkeys(
        AGENT1_TIER_FEATURES + base_features + engineered_features +
        ["applicant_cluster", "program_cluster"])),
    "7. Agent1 Tier + real-calibrated cutoffs (6c2)": list(dict.fromkeys(
        AGENT1_TIER_FEATURES + base_features_v2 + engineered_features_v2)),
}

# Hard guarantee: no candidate is allowed to drop Agent 1 Tier, regardless of what
# automated feature selection might otherwise try to do.
for _name, _feats in feature_set_candidates.items():
    _missing = [c for c in AGENT1_TIER_FEATURES if c not in _feats]
    assert not _missing, f"Agent 1 Tier feature(s) {_missing} missing from candidate '{_name}' -- not allowed."

cv_results = {}
for name, feats in feature_set_candidates.items():
    clf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42,
                                  class_weight="balanced", n_jobs=-1)
    scores = cross_val_score(clf, m.loc[train_idx, feats], y.loc[train_idx], cv=skf, scoring="accuracy")
    cv_results[name] = (scores.mean(), scores.std())
    print(f"{name:<48} CV accuracy = {scores.mean():.4f} +/- {scores.std():.4f}")

winning_name = max(cv_results, key=lambda k: cv_results[k][0])
FINAL_FEATURES = feature_set_candidates[winning_name]
assert "tier_score" in FINAL_FEATURES, "Agent 1 Tier must be present in the production feature set."
print(f"\nWinning feature set: {winning_name}")
print(f"FINAL_FEATURES ({len(FINAL_FEATURES)}): {FINAL_FEATURES}")
print("Agent 1 Tier (tier_score) is present in every candidate above by construction, so this "
      "selection is only ever choosing what to add ON TOP of it -- it can never end up excluding it.")


1. Agent1 Tier + baseline                        CV accuracy = 0.5849 +/- 0.0260
2. Agent1 Tier + engineered features             CV accuracy = 0.5916 +/- 0.0163
3. Agent1 Tier + applicant_cluster               CV accuracy = 0.5821 +/- 0.0182
4. Agent1 Tier + program_cluster                 CV accuracy = 0.5898 +/- 0.0190
5. Agent1 Tier + both clusters                   CV accuracy = 0.5824 +/- 0.0188
6. Agent1 Tier + engineered + both clusters      CV accuracy = 0.5928 +/- 0.0159
7. Agent1 Tier + real-calibrated cutoffs (6c2)   CV accuracy = 0.5856 +/- 0.0144

Winning feature set: 6. Agent1 Tier + engineered + both clusters
FINAL_FEATURES (19): ['tier_score', 'gpa_numeric', 'gre_numeric', 'gre_missing', 'min_gpa', 'min_gre_quant', 'program_admit_rate', 'university_admission_rate', 'gpa_gap', 'gre_gap', 'gpa_ratio', 'gre_ratio', 'gpa_x_tier', 'academic_strength_score', 'selectivity_gap', 'ranking_gap', 'applicant_vs_difficulty', 'applicant_cluster', 'program_cluster']
Agent 1 Tier (tie

### 6h. Agent 1 Tier — feature preparation & consistency check (Part 8)
Agent 1's tier (`tier_score`, joined into every row from `agent2_final_2tier.csv`) is mandatory in every candidate in Section 6g and cannot be dropped by feature selection — the assertions above enforce that mechanically, not just by convention. This section locks in exactly how `tier_score` is represented (so training and live inference can never drift apart), fails loudly if any row is missing it, and documents which saved clustering artifacts inference must reuse for any cluster feature that made it into `FINAL_FEATURES`.

*(An earlier exploratory pass tried an additional applicant-strength composite signal alongside the mandatory tier — see the note in the code cell below for why it was dropped and why it was never used to decide whether to keep `tier_score`, which is never optional.)*


In [19]:
# --- Agent 1 Tier: consistency check + the exact encoding used at train AND inference time ---
assert "tier_score" in FINAL_FEATURES, (
    "Agent 1 Tier (tier_score) must be part of the production feature set -- "
    "it is not optional and must never be removed by feature selection."
)

# tier_score is used as-is: a continuous ~0-100 competitiveness score that is Agent 1's
# output, already joined into every row. No re-binning/re-encoding happens anywhere, so
# train-time and inference-time representations are identical by construction. This
# constant documents that representation so inference code (Section 8) can assert
# against it instead of silently assuming it.
AGENT1_TIER_REPRESENTATION = {
    "column": "tier_score",
    "dtype": "float",
    "valid_range": (float(m["tier_score"].min()), float(m["tier_score"].max())),
    "source": "agent2_final_2tier.csv (Agent 1's university/program tier output)",
}

invalid_tier_rows = int(m["tier_score"].isna().sum())
if invalid_tier_rows:
    raise ValueError(
        f"{invalid_tier_rows} rows have a missing/invalid Agent 1 tier_score after the merge. "
        "Agent 2 will not silently default these -- fix the upstream join before training."
    )
print("Agent 1 Tier representation locked in:", AGENT1_TIER_REPRESENTATION)

# Applicant/university clustering features, if selected into FINAL_FEATURES, reuse the
# exact scaler + KMeans model fit in Section 6e -- nothing is refit here, so train-time
# and inference-time cluster assignments are guaranteed to be produced the same way.
if "applicant_cluster" in FINAL_FEATURES:
    print(f"applicant_cluster is in FINAL_FEATURES -- inference must reuse "
          f"`applicant_scaler` / `applicant_kmeans` fit on {APPLICANT_CLUSTER_FEATURES} in Section 6e.")
if "program_cluster" in FINAL_FEATURES:
    print(f"program_cluster is in FINAL_FEATURES -- inference must reuse "
          f"`university_cluster_scaler` / `university_cluster_model` fit on {UNIV_CLUSTER_COLS} in Section 6e.")

# Historical note: an earlier version of this section tried an additional applicant-strength
# composite signal (built from gpa_gap/gre_gap, standing in for richer per-applicant data
# Agent 1 doesn't have yet) ALONGSIDE the mandatory tier_score, to see whether it added
# anything on top of tier_score + engineered features. It measured no reliable lift and is
# not part of the production feature set. Importantly, it was never used to decide whether
# to keep tier_score itself -- tier_score is never optional -- so it's omitted here rather
# than kept in a form that could be misread as an "Agent 1 tier on/off" test, which this
# architecture explicitly forbids.


Agent 1 Tier representation locked in: {'column': 'tier_score', 'dtype': 'float', 'valid_range': (59.79, 91.5), 'source': "agent2_final_2tier.csv (Agent 1's university/program tier output)"}
applicant_cluster is in FINAL_FEATURES -- inference must reuse `applicant_scaler` / `applicant_kmeans` fit on ['gpa_numeric', 'gre_numeric'] in Section 6e.
program_cluster is in FINAL_FEATURES -- inference must reuse `university_cluster_scaler` / `university_cluster_model` fit on ['us_news_ranking', 'world_ranking', 'tuition_out_state_usd', 'program_admit_rate', 'university_admission_rate', 'tier_score', 'min_gpa', 'min_gre_quant', 'student_size'] in Section 6e.


### 6i. Model comparison, tuning, and calibration (Parts 7 and 9)
Eight model families compared by 5-fold CV on the training portion; the winner is tuned via `RandomizedSearchCV` (still training-only) and calibrated; the test set is touched exactly once, below, for the final numbers.

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
import lightgbm as lgb

try:
    import catboost as cb
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("catboost not installed -- skipping (pip install catboost to include it)")

assert "tier_score" in FINAL_FEATURES, "Agent 1 Tier missing from FINAL_FEATURES -- every model below must receive it."

# The architecture specifies RandomForestClassifier as Agent 2's PRODUCTION admission model
# (see the framework doc). The comparison below is kept for research/monitoring visibility --
# it tells you whether RF is leaving CV accuracy on the table -- but it does not choose the
# deployed model. See the note printed after the loop for exactly what is and isn't overridden.
PRODUCTION_MODEL_NAME = "RandomForest"

candidates = {
    "LogisticRegression": Pipeline([("scaler", StandardScaler()),
                                     ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42,
                                            class_weight="balanced", n_jobs=-1),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=300, max_depth=8, random_state=42,
                                        class_weight="balanced", n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(random_state=42, n_estimators=200, max_depth=3),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42, class_weight="balanced"),
    "XGBoost": xgb.XGBClassifier(random_state=42, eval_metric="logloss", n_estimators=200, max_depth=4),
    "LightGBM": lgb.LGBMClassifier(random_state=42, n_estimators=200, max_depth=4, verbose=-1),
}
if HAS_CATBOOST:
    candidates["CatBoost"] = cb.CatBoostClassifier(random_state=42, iterations=200, depth=4, verbose=False)

Xtr, ytr = m.loc[train_idx, FINAL_FEATURES], y.loc[train_idx]
model_cv_scores = {}
model_cv_f1 = {}
model_cv_precision = {}
model_cv_recall = {}
for name, clf in candidates.items():
    acc_scores = cross_val_score(clf, Xtr, ytr, cv=skf, scoring="accuracy", n_jobs=1)
    f1_scores = cross_val_score(clf, Xtr, ytr, cv=skf, scoring="f1", n_jobs=1)
    prec_scores = cross_val_score(clf, Xtr, ytr, cv=skf, scoring="precision", n_jobs=1)
    rec_scores = cross_val_score(clf, Xtr, ytr, cv=skf, scoring="recall", n_jobs=1)
    model_cv_scores[name] = acc_scores.mean()
    model_cv_f1[name] = f1_scores.mean()
    model_cv_precision[name] = prec_scores.mean()
    model_cv_recall[name] = rec_scores.mean()
    print(f"{name:<22} CV accuracy = {acc_scores.mean():.4f} +/- {acc_scores.std():.4f}  "
          f"| F1 = {f1_scores.mean():.4f}  precision = {prec_scores.mean():.4f}  recall = {rec_scores.mean():.4f}")

best_model_name = max(model_cv_scores, key=model_cv_scores.get)  # research-comparison winner, NOT necessarily deployed
print(f"\nBest by CV accuracy (research comparison only): {best_model_name} "
      f"({model_cv_scores[best_model_name]:.4f}, F1={model_cv_f1[best_model_name]:.4f})")

if best_model_name != PRODUCTION_MODEL_NAME:
    gap = model_cv_scores[best_model_name] - model_cv_scores[PRODUCTION_MODEL_NAME]
    print(f"NOTE: the architecture specifies RandomForestClassifier as Agent 2's PRODUCTION model. "
          f"Production will use '{PRODUCTION_MODEL_NAME}' (CV accuracy "
          f"{model_cv_scores[PRODUCTION_MODEL_NAME]:.4f}) rather than the CV-best '{best_model_name}' "
          f"(CV accuracy {model_cv_scores[best_model_name]:.4f}); gap = {gap:+.4f}. The comparison "
          f"above is retained for visibility, not to silently override the architecture's model choice.")
else:
    print("RandomForest is also the CV-best model here -- the architecture requirement and the "
          "empirical comparison agree, so nothing is being overridden.")


catboost not installed -- skipping (pip install catboost to include it)
LogisticRegression     CV accuracy = 0.5986 +/- 0.0245  | F1 = 0.6260  precision = 0.6279  recall = 0.6247
RandomForest           CV accuracy = 0.5928 +/- 0.0159  | F1 = 0.6208  precision = 0.6230  recall = 0.6192
ExtraTrees             CV accuracy = 0.5946 +/- 0.0146  | F1 = 0.6143  precision = 0.6298  recall = 0.6002
GradientBoosting       CV accuracy = 0.5809 +/- 0.0094  | F1 = 0.6381  precision = 0.5963  recall = 0.6867
HistGradientBoosting   CV accuracy = 0.5741 +/- 0.0145  | F1 = 0.5983  precision = 0.6077  recall = 0.5900
XGBoost                CV accuracy = 0.5609 +/- 0.0108  | F1 = 0.6008  precision = 0.5883  recall = 0.6141
LightGBM               CV accuracy = 0.5689 +/- 0.0082  | F1 = 0.6207  precision = 0.5898  recall = 0.6552

Best by CV accuracy (research comparison only): LogisticRegression (0.5986, F1=0.6260)
NOTE: the architecture specifies RandomForestClassifier as Agent 2's PRODUCTION model. Prod

In [21]:
param_dists = {
    "LogisticRegression": {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100]},
    "RandomForest": {"n_estimators": [200, 300, 400, 600], "max_depth": [4, 6, 8, 10, 12, None],
                      "min_samples_leaf": [1, 2, 4, 8], "max_features": ["sqrt", "log2", None]},
    "ExtraTrees": {"n_estimators": [200, 300, 400, 600], "max_depth": [4, 6, 8, 10, None],
                    "min_samples_leaf": [1, 2, 4, 8]},
    "GradientBoosting": {"n_estimators": [100, 200, 300], "max_depth": [2, 3, 4], "learning_rate": [0.02, 0.05, 0.1]},
    "HistGradientBoosting": {"max_depth": [3, 4, 5, 6, None], "learning_rate": [0.02, 0.05, 0.1, 0.15],
                               "max_iter": [100, 150, 200, 300]},
    "XGBoost": {"n_estimators": [100, 200, 300, 400], "max_depth": [3, 4, 5, 6],
                 "learning_rate": [0.02, 0.05, 0.1, 0.15], "subsample": [0.7, 0.85, 1.0],
                 "colsample_bytree": [0.7, 0.85, 1.0]},
    "LightGBM": {"n_estimators": [100, 200, 300, 400], "max_depth": [3, 4, 5, 6, -1],
                  "learning_rate": [0.02, 0.05, 0.1, 0.15], "num_leaves": [15, 31, 63]},
    "CatBoost": {"iterations": [100, 200, 300], "depth": [3, 4, 5, 6], "learning_rate": [0.02, 0.05, 0.1]},
}

assert "tier_score" in FINAL_FEATURES, "Agent 1 Tier missing from FINAL_FEATURES -- tuning must not drop it."
assert "tier_score" in Xtr.columns, "Agent 1 Tier missing from the training matrix used for tuning."

# Tune the PRODUCTION model (RandomForest, per the architecture), not necessarily whichever
# model won the CV comparison above -- see the note printed in Section 6i for the exact gap.
search = RandomizedSearchCV(candidates[PRODUCTION_MODEL_NAME], param_distributions=param_dists[PRODUCTION_MODEL_NAME],
                             n_iter=25, cv=skf, scoring="accuracy", random_state=42, n_jobs=-1)
search.fit(Xtr, ytr)
tuned_model = search.best_estimator_
print(f"Tuning production model: {PRODUCTION_MODEL_NAME}")
print("Best params:", search.best_params_)
print("Best CV accuracy:", search.best_score_)

# Calibrated via internal CV on the training portion only -- test set (and the separate
# val_idx used for threshold selection in 6j) are still untouched at this point.
admit_model = CalibratedClassifierCV(tuned_model, method="sigmoid", cv=5)
admit_model.fit(Xtr, ytr)


Tuning production model: RandomForest
Best params: {'n_estimators': 600, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 6}
Best CV accuracy: 0.5978077843995242


CalibratedClassifierCV(cv=5,
                       estimator=RandomForestClassifier(class_weight='balanced',
                                                        max_depth=6,
                                                        max_features='log2',
                                                        min_samples_leaf=4,
                                                        n_estimators=600,
                                                        n_jobs=-1,
                                                        random_state=42))

### 6j. Final test-set evaluation and 3-class comparison (Part 6)
The test set is used exactly once, here.

In [22]:
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                              brier_score_loss, classification_report, confusion_matrix)

# --- Threshold selection on the VALIDATION set only (val_idx from Section 6f) ---
# The test set below is still touched exactly once, after this cell picks a threshold.
# Caveat, stated plainly: val_idx is a subset of train_idx, and train_idx (not train_idx2)
# was the pool used for the CV-based model/hyperparameter search in 6g-6i, so this is a
# best-effort, slightly conservative proxy for a fully held-out validation set -- it was
# never used to fit the final estimator's parameters or decision boundary directly, and
# it is never touched again after this cell picks the threshold.
Xval, yval = m.loc[val_idx, FINAL_FEATURES], y.loc[val_idx]
val_proba = admit_model.predict_proba(Xval)[:, list(admit_model.classes_).index(1)]

threshold_grid = np.arange(0.05, 0.96, 0.01)
threshold_f1 = [(t, f1_score(yval, (val_proba >= t).astype(int))) for t in threshold_grid]
best_threshold, best_val_f1 = max(threshold_f1, key=lambda p: p[1])
default_val_f1 = f1_score(yval, (val_proba >= 0.5).astype(int))
print(f"Validation-selected threshold: {best_threshold:.2f}  (val F1={best_val_f1:.4f}, "
      f"vs. default 0.5 threshold val F1={default_val_f1:.4f})")

# --- Final test-set evaluation (touched exactly once, right here) ---
Xte, yte = m.loc[test_idx, FINAL_FEATURES], y.loc[test_idx]
proba = admit_model.predict_proba(Xte)[:, list(admit_model.classes_).index(1)]
preds_default = (proba >= 0.5).astype(int)
preds = (proba >= best_threshold).astype(int)


def _report(label, preds_arr):
    metrics = {
        "Accuracy": accuracy_score(yte, preds_arr), "F1": f1_score(yte, preds_arr),
        "Precision": precision_score(yte, preds_arr), "Recall": recall_score(yte, preds_arr),
        "ROC-AUC": roc_auc_score(yte, proba), "Brier": brier_score_loss(yte, proba),
    }
    print(f"\n=== FINAL TEST METRICS ({label}) ===")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
    return metrics


final_metrics_default_threshold = _report("threshold=0.50, default", preds_default)
final_metrics = _report(f"threshold={best_threshold:.2f}, validation-selected", preds)
print("\nClassification report (validation-selected threshold):\n", classification_report(yte, preds))
print("Confusion matrix (validation-selected threshold):\n", confusion_matrix(yte, preds))

# --- Does Agent 2 add value beyond Agent 1's tier alone? (test set, touched once above) ---
tier_only_clf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42,
                                        class_weight="balanced", n_jobs=-1)
tier_only_clf.fit(m.loc[train_idx, AGENT1_TIER_FEATURES], y.loc[train_idx])
tier_only_preds = tier_only_clf.predict(m.loc[test_idx, AGENT1_TIER_FEATURES])
tier_only_proba = tier_only_clf.predict_proba(m.loc[test_idx, AGENT1_TIER_FEATURES])[
    :, list(tier_only_clf.classes_).index(1)]
tier_only_acc = accuracy_score(yte, tier_only_preds)
tier_only_auc = roc_auc_score(yte, tier_only_proba)

print("\n=== Agent 1 Tier alone vs. Agent 1 Tier + Agent 2 features (test set) ===")
print(f"Agent 1 Tier alone:            accuracy={tier_only_acc:.4f}  ROC-AUC={tier_only_auc:.4f}")
print(f"Agent 1 Tier + Agent 2 (full): accuracy={final_metrics['Accuracy']:.4f}  ROC-AUC={final_metrics['ROC-AUC']:.4f}")
print(f"Delta from Agent 2's additional features: accuracy {final_metrics['Accuracy']-tier_only_acc:+.4f}, "
      f"ROC-AUC {final_metrics['ROC-AUC']-tier_only_auc:+.4f}")

# --- 3-class comparison (Part 6) ---
m["y_3class"] = m["admit_result"].map({"Reject": 0, "Waitlist": 1, "Admit": 2})
y3 = m["y_3class"]
clf3 = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight="balanced", n_jobs=-1)
cv3 = cross_val_score(clf3, m.loc[train_idx, FINAL_FEATURES], y3.loc[train_idx], cv=skf, scoring="accuracy")
print(f"\n3-class CV accuracy: {cv3.mean():.4f} +/- {cv3.std():.4f}  (binary CV, production model "
      f"{PRODUCTION_MODEL_NAME}, was {model_cv_scores[PRODUCTION_MODEL_NAME]:.4f})")
clf3.fit(m.loc[train_idx, FINAL_FEATURES], y3.loc[train_idx])
preds3 = clf3.predict(Xte)
print("3-class test accuracy:", accuracy_score(y3.loc[test_idx], preds3))
print(classification_report(y3.loc[test_idx], preds3, target_names=["Reject", "Waitlist", "Admit"]))
print("\nWaitlist sits genuinely between Reject and Admit in mean gpa_gap (checked separately) --")
print("it is a real ordinal middle class, not noise. But splitting it out costs several points of")
print("accuracy for a use case (counseling on admit likelihood) where binary Admit/Not-Admit is the")
print("more directly actionable answer. BINARY is used as the production formulation; Waitlist is")
print("folded into Not-Admit, documented explicitly here rather than silently.")


Validation-selected threshold: 0.38  (val F1=0.7230, vs. default 0.5 threshold val F1=0.7064)

=== FINAL TEST METRICS (threshold=0.50, default) ===
Accuracy: 0.6125
F1: 0.6750
Precision: 0.6159
Recall: 0.7468
ROC-AUC: 0.6573
Brier: 0.2296

=== FINAL TEST METRICS (threshold=0.38, validation-selected) ===
Accuracy: 0.5867
F1: 0.7063
Precision: 0.5722
Recall: 0.9224
ROC-AUC: 0.6573
Brier: 0.2296

Classification report (validation-selected threshold):
               precision    recall  f1-score   support

           0       0.68      0.19      0.30       463
           1       0.57      0.92      0.71       541

    accuracy                           0.59      1004
   macro avg       0.63      0.56      0.50      1004
weighted avg       0.62      0.59      0.52      1004

Confusion matrix (validation-selected threshold):
 [[ 90 373]
 [ 42 499]]

=== Agent 1 Tier alone vs. Agent 1 Tier + Agent 2 features (test set) ===
Agent 1 Tier alone:            accuracy=0.5249  ROC-AUC=0.5058
Agent 1 

### 6k. Summary table and the honest verdict

| Stage | Features | CV / Test Accuracy |
|---|---|---|
| 1. Agent 1 Tier + baseline | 5 | 58.5% (CV) |
| 2. Agent 1 Tier + engineered features | 17 | 58.7% (CV) |
| 3. Agent 1 Tier + applicant cluster | 5 | within noise of baseline (CV) |
| 4. Agent 1 Tier + program cluster | 5 | 58.9% (CV) -- best simple cluster variant, still marginal |
| 5. Agent 1 Tier + both clusters | 6 | within noise of #4 (CV) |
| 6. Agent 1 Tier + engineered + both clusters | 19 | see printed run for exact value |
| 7. Agent 1 Tier + real-calibrated cutoffs (6c2) | 17 | +0.37pp for the linear model, -0.95pp for RandomForest -- mixed, not a clear win |
| **Final model (tuned + calibrated, winning set above)** | **varies** | **~60-61% (held-out test, touched once)** |

*(Agent 1 Tier — `tier_score` — is present in every row of this table by construction; the comparison is only ever about what to add on top of it, never whether to include it. The exact numbers for your run print directly above these cells when you run them in Colab against your two uploaded CSVs.)*

**Verdict: 85% is not reached, and cannot be legitimately reached on this data as it stands.** Every lever in Parts 2-8 was tried honestly -- richer catalog columns, 9 engineered features, real-world-anchored recalibration of the program cutoffs using cited published admissions data for MIT/Georgia Tech/Delaware, two clustering approaches, up to 8 model families, hyperparameter tuning, calibration, and Agent 1's mandatory tier throughout -- and the ceiling stayed at ~59-61% throughout. The real-data recalibration in particular is a useful negative result: it confirms the ceiling is about how much the *target itself* is determined by anything measurable in this data, not about any one feature being miscalibrated. That is not a modeling failure; a depth-3 decision tree using only `gpa_gap`/`gre_gap` alone tops out at ~60% *training* accuracy too (checked directly) -- the data itself only weakly determines the outcome.

**Why, specifically:**
1. **GPA/GRE are banded, not continuous** (e.g. `"3.4-3.5"`), throwing away exactly the fine-grained differences most likely to separate close cases.
2. **The program catalog's own cutoffs are unrealistically narrow** -- `min_gpa` ranges only 3.00-3.20 and `min_gre_quant` only 148-161 across all 4,500 programs (checked directly), so "distance from cutoff" barely varies program-to-program.
3. **No real per-applicant differentiators exist beyond GPA/GRE** -- no undergrad tier, work experience, research, publications, SOP/LOR content. These are exactly the fields a real admissions decision depends on most, and none of them are in this admits history.
4. **The dataset is explicitly synthetic** (`data_quality: synthetic_demo` on every original row) -- the *generating process* behind `admit_result` isn't fully determined by any visible column, so there is a hard information ceiling no amount of modeling can cross.

**Exactly what would close the gap** (Part 2's "structure the notebook so I can add it later"): continuous (unbanded) GPA/GRE per applicant; real undergrad-institution tier; work experience (months); research experience and publication count; SOP/LOR scores (Agent 7's `profile_strength_score`, once it is populated for real historical applicants rather than defaulted to `0.5`); and program cutoffs with realistic real-world spread rather than a 0.2-point range. Every code path in this section (`base_features`, `engineered_features`, `compute_tier_ml`) is written to accept these the moment they exist -- no restructuring needed, just wider CSVs and, eventually, a real Agent 1 model artifact instead of `tier_score` alone.


## 7. SHAP explainability
The winning model in Section 6i isn't fixed to one family (it's whichever of the 8 candidates scores highest by CV — a linear model in the run behind Section 6k's numbers, but that can change if you re-run this against different data). `shap.TreeExplainer` only works for tree models, so a model-agnostic `shap.Explainer` is used instead — it wraps `predict_proba` directly and works the same way regardless of which model family won.

In [23]:
import shap

# Model-agnostic: wraps predict_proba directly, so this works whether tuned_model ended
# up being linear (LogisticRegression) or tree-based -- no assumption about model family.
background = shap.sample(Xtr, 100, random_state=42)
shap_explainer = shap.Explainer(tuned_model.predict_proba, background)

sample_shap = shap_explainer(Xte.iloc[[0]])
print("SHAP values shape:", sample_shap.values.shape)

POSITIVE_CLASS_INDEX = list(tuned_model.classes_).index(1) if hasattr(tuned_model, "classes_") else 1
print("Positive class (admit=1) at index:", POSITIVE_CLASS_INDEX)


def get_shap_top_factors(X_row_df, top_n=3):
    values = shap_explainer(X_row_df)
    vals = values.values[0][:, POSITIVE_CLASS_INDEX] if values.values.ndim == 3 else values.values[0]
    pairs = sorted(zip(X_row_df.columns, vals), key=lambda p: abs(p[1]), reverse=True)
    return [{"feature": f, "impact": round(float(v), 4)} for f, v in pairs[:top_n]]


print("\nTop factors for one test row:", get_shap_top_factors(Xte.iloc[[0]]))

PermutationExplainer explainer: 2it [00:10, 10.54s/it]               


SHAP values shape: (1, 19, 2)
Positive class (admit=1) at index: 1

Top factors for one test row: [{'feature': 'gpa_numeric', 'impact': -0.0479}, {'feature': 'gpa_x_tier', 'impact': -0.0333}, {'feature': 'gpa_gap', 'impact': -0.0259}]


In [24]:
print("admit_model (calibrated):", type(admit_model))
print("tuned_model (uncalibrated, used for SHAP):", type(tuned_model))
print("shap_explainer:", type(shap_explainer))
print("university_cluster_model:", type(university_cluster_model))

admit_model (calibrated): <class 'sklearn.calibration.CalibratedClassifierCV'>
tuned_model (uncalibrated, used for SHAP): <class 'sklearn.ensemble._forest.RandomForestClassifier'>
shap_explainer: <class 'shap.explainers._permutation.PermutationExplainer'>
university_cluster_model: <class 'sklearn.cluster._kmeans.KMeans'>


In [25]:
import joblib

joblib.dump(admit_model, "agent2_admission_model.joblib")               # final calibrated model used for prediction
joblib.dump(FINAL_FEATURES, "agent2_feature_columns.joblib")             # exact column order the model expects
joblib.dump(AGENT1_TIER_FEATURES, "agent2_agent1_tier_features.joblib")  # which columns are Agent 1's mandatory tier input
joblib.dump(AGENT1_TIER_REPRESENTATION, "agent2_agent1_tier_representation.joblib")  # encoding/valid range, checked at inference
joblib.dump(best_threshold, "agent2_decision_threshold.joblib")          # validation-selected probability threshold

joblib.dump(university_cluster_model, "university_cluster_model.joblib")
joblib.dump(university_cluster_scaler, "university_cluster_scaler.joblib")
joblib.dump(UNIV_CLUSTER_COLS, "university_cluster_features.joblib")
joblib.dump(univ_cluster_medians, "university_cluster_medians.joblib")

# Applicant clustering artifacts are saved even though the winning feature set above may not
# need applicant_cluster -- per the brief, they must be saved regardless so live inference (or
# a future retrain) can always produce an applicant_cluster value without refitting from scratch.
joblib.dump(applicant_kmeans, "applicant_cluster_model.joblib")
joblib.dump(applicant_scaler, "applicant_cluster_scaler.joblib")
joblib.dump(applicant_cluster_medians, "applicant_cluster_medians.joblib")
joblib.dump(APPLICANT_CLUSTER_FEATURES, "applicant_cluster_features.joblib")

joblib.dump(shap_explainer, "agent2_shap_explainer.joblib")
# No separate agent2_scaler.joblib: when the winning model is LogisticRegression its
# StandardScaler is bundled inside the Pipeline (and therefore inside admit_model) already;
# a standalone scaler is only needed if you swap in a model that doesn't embed one.

from google.colab import files

_artifact_files = [
    "agent2_admission_model.joblib", "agent2_feature_columns.joblib",
    "agent2_agent1_tier_features.joblib", "agent2_agent1_tier_representation.joblib",
    "agent2_decision_threshold.joblib",
    "university_cluster_model.joblib", "university_cluster_scaler.joblib",
    "university_cluster_features.joblib", "university_cluster_medians.joblib",
    "applicant_cluster_model.joblib", "applicant_cluster_scaler.joblib",
    "applicant_cluster_medians.joblib", "applicant_cluster_features.joblib",
    "agent2_shap_explainer.joblib",
]
for f in _artifact_files:
    files.download(f)

print(f"Saved and downloaded {len(_artifact_files)} production artifacts -- Agent 1 tier "
      "metadata, both clustering pipelines (always saved), the validation-selected decision "
      "threshold, and the admission model are all included.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloaded 14 production artifacts -- Agent 1 tier metadata, both clustering pipelines (always saved), the validation-selected decision threshold, and the admission model are all included.


## 8. Tier + admit probability at inference time
Builds the same features `FINAL_FEATURES` names, from the live candidate's profile and the retrieved program's payload, and scores them through the calibrated `admit_model`. Agent 1's tier (`tier_score`) is read directly from the payload and validated before anything else runs -- if it is missing, Agent 2 refuses the prediction rather than inventing a value (`Agent1TierMissingError`). `program_cluster` and `applicant_cluster` are both computed on the fly via the saved clustering scalers/models from Section 6e (as DataFrames with the training-time column names and order, so `StandardScaler` never warns about missing feature names) -- whichever of the two actually appears in `FINAL_FEATURES` is used, but both are always computed so the feature-row build never depends on which combination won in Section 6g. `get_agent7_profile_strength` performs the equivalent check for Agent 7's `profile_strength_score` (`Agent7ScoreMissingError` if missing in integrated mode, a visibly-logged neutral default only in `standalone_mode=True`), and normalizes it from Agent 7's 0-100 scale to the 0-1 fraction `compute_final_score` expects.


In [26]:
class Agent1TierMissingError(ValueError):
    """Raised when Agent 1 has not produced a tier for this candidate. Agent 2 must
    stop rather than invent a default tier -- see the architecture requirement."""
    pass


class Agent7ScoreMissingError(ValueError):
    """Raised when Agent 7's profile_strength_score is required (integrated pipeline mode)
    but absent from state. Agent 2 must stop rather than silently default it -- see the
    architecture requirement in Section 5/8. Not raised in standalone_mode (Section 9)."""
    pass


def get_required_agent1_tier(payload):
    """Agent 1's tier (tier_score) is a REQUIRED input -- Agent 2 never fabricates it."""
    tier_score = payload.get("tier_score", None)
    if tier_score is None or tier_score == "":
        raise Agent1TierMissingError("Agent 1 tier is required before Agent 2 evaluation.")
    try:
        tier_score = float(tier_score)
    except (ValueError, TypeError):
        raise Agent1TierMissingError("Agent 1 tier is required before Agent 2 evaluation.")
    if pd.isna(tier_score):
        raise Agent1TierMissingError("Agent 1 tier is required before Agent 2 evaluation.")
    lo, hi = AGENT1_TIER_REPRESENTATION["valid_range"]
    if not (lo - 1e-6 <= tier_score <= hi + 1e-6):
        print(f"WARNING: Agent 1 tier_score={tier_score} is outside the training range "
              f"[{lo:.1f}, {hi:.1f}] -- proceeding, but treat this prediction with caution.")
    return tier_score


def estimate_program_cluster(payload):
    row = pd.DataFrame([[float(payload.get(c, 0) or 0) for c in UNIV_CLUSTER_COLS]],
                        columns=UNIV_CLUSTER_COLS)
    scaled = university_cluster_scaler.transform(row)  # DataFrame in -> matches the fitted feature names, no warning
    return int(university_cluster_model.predict(scaled)[0])


def estimate_applicant_cluster(student_gpa, student_gre, gre_missing):
    gre_for_cluster = student_gre if not gre_missing else applicant_cluster_medians["gre_numeric"]
    row = pd.DataFrame([[student_gpa, gre_for_cluster]], columns=APPLICANT_CLUSTER_FEATURES)
    scaled = applicant_scaler.transform(row)
    return int(applicant_kmeans.predict(scaled)[0])


def _extract_gre_quant(profile):
    """Handles both the nested shape Agent 1's profile agent actually produces
    (test_scores: {GRE: {quant: 165}}) and a flat gre_quant field, for compatibility."""
    test_scores = profile.get("test_scores")
    if isinstance(test_scores, dict):
        gre_block = test_scores.get("GRE") or test_scores.get("gre")
        if isinstance(gre_block, dict):
            value = gre_block.get("quant")
            if value not in (None, ""):
                return value
    return profile.get("gre_quant")


def compute_tier_ml(profile, payload, extracurricular_score=None):
    # extracurricular_score is accepted for interface compatibility with the rest of the
    # pipeline (Agent 7 calls this the same way it always has) but is NOT fed into the
    # model below -- Section 6 found it's a constant 0.5 in the training data (Agent 7 has
    # no real historical per-application scores merged into the admits history yet), so a
    # trained model cannot use it. Wire it in once real historical extracurricular_score
    # values exist to retrain on.

    # --- Agent 1 tier is REQUIRED. No default, no fabrication -- fail loudly if missing. ---
    tier_score = get_required_agent1_tier(payload)

    normalized_gpa = normalize_gpa_to_4(profile.get("gpa"), profile.get("gpa_scale"))
    if normalized_gpa is None:
        raise ValueError("A valid applicant GPA is required before Agent 2 evaluation.")
    student_gpa = float(normalized_gpa)

    gre_value = _extract_gre_quant(profile)
    if gre_value is None or gre_value == "":
        student_gre, gre_missing = 0.0, 1
    else:
        student_gre, gre_missing = float(gre_value), 0

    min_gpa = float(payload.get("min_gpa", 3.0) or 3.0)
    min_gre_quant = float(payload.get("min_gre_quant", 0) or 0)
    program_admit_rate = float(payload.get("program_admit_rate", 0.3) or 0.3)
    university_admission_rate = float(payload.get("university_admission_rate", 0.3) or 0.3)
    us_news_ranking = float(payload.get("us_news_ranking", 0) or 0)
    world_ranking = float(payload.get("world_ranking", 0) or 0)

    gpa_gap = student_gpa - min_gpa
    gre_gap = (student_gre - min_gre_quant) if not gre_missing else 0.0
    gpa_ratio = student_gpa / min_gpa if min_gpa else 1.0
    gre_ratio = (student_gre / min_gre_quant) if (min_gre_quant and not gre_missing) else 1.0
    gpa_x_tier = student_gpa * (tier_score / 100)
    academic_strength_score = 0.5 * gpa_gap + 0.5 * (gre_gap / 10)
    selectivity_gap = program_admit_rate - university_admission_rate
    ranking_gap = us_news_ranking - world_ranking
    applicant_vs_difficulty = gpa_gap - (1 - program_admit_rate)

    # Both cluster assignments are always computed from the saved scaler/model used in
    # training, regardless of which one ended up in FINAL_FEATURES -- this keeps inference
    # correct even if a future retrain changes the winning combination, and is what actually
    # fixes the old "KeyError: ['applicant_cluster'] not in index" failure (the previous
    # version of this function only ever built program_cluster).
    program_cluster = estimate_program_cluster(payload)
    applicant_cluster = estimate_applicant_cluster(student_gpa, student_gre, gre_missing)

    feature_row = {
        "tier_score": tier_score,
        "gpa_numeric": student_gpa, "gre_numeric": student_gre, "gre_missing": gre_missing,
        "min_gpa": min_gpa, "min_gre_quant": min_gre_quant,
        "program_admit_rate": program_admit_rate, "university_admission_rate": university_admission_rate,
        "gpa_gap": gpa_gap, "gre_gap": gre_gap, "gpa_ratio": gpa_ratio, "gre_ratio": gre_ratio,
        "gpa_x_tier": gpa_x_tier, "academic_strength_score": academic_strength_score,
        "selectivity_gap": selectivity_gap, "ranking_gap": ranking_gap,
        "applicant_vs_difficulty": applicant_vs_difficulty,
        "program_cluster": program_cluster, "applicant_cluster": applicant_cluster,
    }
    missing_final_features = [c for c in FINAL_FEATURES if c not in feature_row]
    if missing_final_features:
        raise KeyError(f"Inference cannot build FINAL_FEATURES columns {missing_final_features} -- "
                        "compute_tier_ml needs updating to compute them.")
    X_new = pd.DataFrame([feature_row])[FINAL_FEATURES]  # exact column order + names FINAL_FEATURES expects

    positive_idx = list(admit_model.classes_).index(1)
    admit_prob = admit_model.predict_proba(X_new)[0][positive_idx]

    if admit_prob >= 0.65:
        tier = "Safe"
    elif admit_prob >= 0.35:
        tier = "Target"
    else:
        tier = "Reach"

    top_factors = get_shap_top_factors(X_new)
    return tier, admit_prob, top_factors, normalized_gpa, tier_score


## 9. Explanation generation (Groq LLaMA — grounded in SHAP factors + normalized GPA)

In [27]:
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])


def generate_explanation(university_name, program_name, normalized_gpa, target_program, tier, score,
                          top_factors, description_text="", agent1_tier=None, extracurricular_score=None):
    """
    Builds the Groq prompt from three explicitly separated kinds of information, so the LLM
    can't blur what actually moved the admission-probability estimate:
      1. Model-derived factors -- real SHAP contributions from the trained admission model.
      2. Upstream agent context -- Agent 1's tier (already one of the model-derived factors
         above) and Agent 7's profile_strength_score. Agent 7's score is real, but it is NOT
         yet a trained input to the admission-probability model (Section 6/8), so it is framed
         as background on the applicant, not as something that moved the probability.
      3. Retrieved university/program information.
    """
    factors_str = ", ".join(f"{f['feature']} ({'+' if f['impact']>=0 else ''}{f['impact']})" for f in top_factors)

    context_lines = [f"Model-derived factors behind the admission estimate: {factors_str}."]
    if agent1_tier is not None:
        context_lines.append(f"Agent 1's university/program tier score: {agent1_tier:.1f}/100 "
                              f"(already included among the model-derived factors above).")
    if extracurricular_score is not None:
        context_lines.append(
            f"Agent 7's applicant profile-strength score: {extracurricular_score:.1f}/100 -- background "
            f"context on the applicant's extracurricular/SOP strength. It currently informs Agent 2's "
            f"overall match ranking but is NOT yet a trained input to the admission-probability estimate "
            f"above -- do not claim it raised or lowered that probability."
        )
    context_lines.append(f"Retrieved program information: {description_text[:300]}")

    prompt = (
        f"Explain in 2-3 sentences why {university_name}'s {program_name} is a {tier} match for a "
        f"student with GPA {normalized_gpa:.2f}/4.0 targeting {target_program}. Overall match score: {score}. "
        f"Use ONLY the information below -- do not invent reasons it doesn't support, and do not attribute "
        f"the admission probability to anything not listed under model-derived factors.\n"
        + "\n".join(context_lines)
    )

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        max_tokens=400,
        reasoning_effort="low",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


## 10. Persistence — save matches to Postgres

In [28]:
from sqlalchemy import create_engine, text
import json as _json

def save_matches_to_db(student_id, university_matches):
    if not POSTGRES_URL:
        print("POSTGRES_URL not set — skipping DB persistence (in-memory only).")
        return
    engine = create_engine(POSTGRES_URL)
    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS university_matches (
                student_id TEXT,
                university TEXT,
                country TEXT,
                region TEXT,
                type TEXT,
                admit_probability FLOAT,
                overall_score FLOAT,
                reasoning TEXT,
                programs JSONB,
                created_at TIMESTAMP DEFAULT now()
            )
        """))
        for m in university_matches:
            conn.execute(text("""
                INSERT INTO university_matches
                (student_id, university, country, region, type, admit_probability, overall_score, reasoning, programs)
                VALUES (:sid, :uni, :country, :region, :type, :prob, :overall, :reasoning, :programs)
            """), {
                "sid": student_id, "uni": m["university"], "country": m["country"],
                "region": m["region"], "type": m["type"], "prob": m["admit_probability"],
                "overall": m["overall_score"], "reasoning": m["reasoning"], "programs": _json.dumps(m["programs"])
            })
    print(f"Saved {len(university_matches)} university matches for student {student_id} to Postgres.")

## 11. Full agent function
University ranking now uses a combined `overall_score` (mean program `match_score` at that university + best `admit_probability`, weighted), not admit_probability alone — a university that's a much better semantic/profile fit can outrank one with a marginally higher admit probability. Program-level detail stays nested under `programs` for Agent 3.

In [29]:
def get_agent7_profile_strength(state, standalone_mode=False):
    """
    Agent 7's profile_strength_score is a 0-100 applicant-strength signal (extracurricular
    evidence blended with SOP/LOR quality -- see Agent 7's `combine_scores`). Agent 2
    normalizes it to a 0-1 fraction for its own weighted matching score (Section 5).

    standalone_mode=False (default): the integrated-pipeline behavior. Agent 7 is a required
    upstream node -- if `state["extracurricular"]["profile_strength_score"]` is missing,
    Agent 2 raises Agent7ScoreMissingError rather than silently defaulting, exactly like
    Agent1TierMissingError for a missing tier_score.

    standalone_mode=True: for developing/testing Agent 2 in isolation, before Agent 7 is
    wired into the running pipeline. Falls back to a neutral 50/100 and prints a visible
    warning so a dev run can never be mistaken for a real Agent 7 score.
    """
    extracurricular_state = state.get("extracurricular")
    raw_score = extracurricular_state.get("profile_strength_score") if isinstance(extracurricular_state, dict) else None

    if raw_score is None or raw_score == "":
        if standalone_mode:
            print("DEV MODE (standalone_mode=True): no Agent 7 output in state -- using a neutral "
                  "default profile_strength_score=50/100 for standalone testing only.")
            raw_score = 50.0
        else:
            raise Agent7ScoreMissingError(
                "Agent 7 profile_strength_score is required before Agent 2 evaluation in the "
                "integrated pipeline. Pass standalone_mode=True only for isolated development/"
                "testing of Agent 2 without Agent 7 wired in."
            )
    try:
        raw_score = float(raw_score)
    except (ValueError, TypeError):
        raise Agent7ScoreMissingError("Agent 7 profile_strength_score must be numeric.")

    normalized = max(0.0, min(1.0, raw_score / 100))
    return raw_score, normalized


def university_matching_agent(state, standalone_mode=False):
    profile = state["profile"]
    preferences = state.get("preferences", {})
    priority = preferences.get("priority", "default")

    # Agent 7's score, like Agent 1's tier, is a required upstream input in the integrated
    # pipeline (standalone_mode=False, the default). standalone_mode=True exists only for
    # developing/testing Agent 2 before Agent 7 is wired in -- see the docstring above.
    extracurricular_score_raw, extracurricular_score = get_agent7_profile_strength(state, standalone_mode)

    candidates = retrieve_candidates(profile, preferences, top_k=20)
    query_text = f"{profile['target_degree']} {profile['target_program']}"
    reranked = rerank(query_text, candidates, top_n=10)

    # Score each program-level candidate first (retains program granularity internally)
    program_level_matches = []
    for candidate, sem_score in reranked:
        payload = candidate.payload
        # extracurricular_score (0-1, normalized above) is a real input to the MATCHING score
        # here -- it is NOT yet a trained input to the admission-probability model below (see
        # Section 6/8 for why: zero variance in the historical training data).
        score = compute_final_score(payload, profile, sem_score, extracurricular_score, priority)
        # Agent 1's tier is required -- if a candidate's payload somehow lacks it, Agent 2
        # skips that one candidate rather than fabricating a tier for it.
        try:
            tier, admit_prob, top_factors, normalized_gpa, agent1_tier = compute_tier_ml(
                profile, payload, extracurricular_score)
        except Agent1TierMissingError as e:
            print(f"Skipping {payload.get('university_name','Unknown')} / "
                  f"{payload.get('program_name','Unknown')}: {e}")
            continue
        explanation = generate_explanation(
            payload["university_name"], payload["program_name"], normalized_gpa,
            profile["target_program"], tier, score, top_factors, payload.get("description_text", ""),
            agent1_tier=agent1_tier, extracurricular_score=extracurricular_score_raw,
        )
        program_level_matches.append({
            "university_name": payload["university_name"],
            "program_name": payload["program_name"],
            "country": payload.get("country", "Unknown"),
            "region": payload.get("region", "Unknown"),
            "type": payload.get("institution_type", "Unknown"),
            "agent1_tier": round(float(agent1_tier), 2),
            "extracurricular_score": round(float(extracurricular_score_raw), 1),
            "tier": tier,
            "match_score": score,
            "admit_probability": round(float(admit_prob), 3),
            "within_stated_budget": tag_budget_signal(payload, preferences),
            "explainability": top_factors,
            "explanation": explanation
        })

    # Group into university-level shortlist (framework's expected output shape)
    by_university = {}
    for m in program_level_matches:
        key = m["university_name"]
        if key not in by_university:
            by_university[key] = {
                "university": m["university_name"],
                "country": m["country"],
                "region": m["region"],
                "type": m["type"],
                "programs": [],
            }
        by_university[key]["programs"].append(m)

    university_matches = []
    for uni in by_university.values():
        programs_here = uni["programs"]
        best_program = max(programs_here, key=lambda p: p["admit_probability"])
        avg_match_score = sum(p["match_score"] for p in programs_here) / len(programs_here)

        uni["admit_probability"] = best_program["admit_probability"]
        uni["agent1_tier"] = best_program["agent1_tier"]
        uni["extracurricular_score"] = best_program["extracurricular_score"]
        # Combined ranking signal: university match quality + admission likelihood,
        # so a strong semantic/profile fit isn't buried under a merely higher admit_probability.
        uni["overall_score"] = round(0.6 * avg_match_score + 0.4 * best_program["admit_probability"], 3)
        uni["reasoning"] = best_program["explanation"]
        university_matches.append(uni)

    university_matches = sorted(university_matches, key=lambda u: u["overall_score"], reverse=True)

    state["matched_universities"] = university_matches
    state["status"] = "university_done"

    save_matches_to_db(state.get("student_id", "unknown"), university_matches)
    return state


## 12. Test run

In [30]:
MOCK_STATE = {
    "student_id": "test-001",
    "profile": {
        "gpa": 8.6, "gpa_scale": 10.0,   # deliberately non-4.0 scale to verify the GPA fix
        "target_program": "Computer Science", "target_degree": "MS",
        "test_scores": {"GRE": {"quant": 165}},  # nested shape Agent 1's profile agent actually produces
    },
    "preferences": {"budget_max_usd": 45000, "priority": "research"},
    "extracurricular": {"profile_strength_score": 80},  # Agent 7's real 0-100 scale, not 0-1
}

print("=" * 70)
print("TEST 7 (run first, others build on it) -- full existing flow still works")
print("RAG, reranking, admission prediction, SHAP, Groq reasoning, ranking, DB persistence")
print("=" * 70)
result = university_matching_agent(MOCK_STATE)
assert result["matched_universities"], "No matches returned -- cannot verify the flow."
assert result["status"] == "university_done"
for u in result["matched_universities"]:
    print(f"{u['university']} ({u['type']}, {u['region']}, {u['country']}) "
          f"— overall_score={u['overall_score']} admit_probability={u['admit_probability']}")
    print(f"  reasoning: {u['reasoning']}")
    for p in u["programs"]:
        print(f"    - {p['program_name']} | agent1_tier={p['agent1_tier']} "
              f"extracurricular_score={p['extracurricular_score']} -> agent2_tier={p['tier']} "
              f"match_score={p['match_score']} within_budget={p['within_stated_budget']}")
    print()
print("PASSED: RAG/rerank/admission/SHAP/Groq/ranking/persistence all completed without error.\n")

print("=" * 70)
print("TEST 1 -- Agent 1 tier is mandatory")
print("=" * 70)
payload_missing_tier = {
    "min_gpa": 3.4, "min_gre_quant": 160, "program_admit_rate": 0.2,
    "university_admission_rate": 0.25, "us_news_ranking": 10, "world_ranking": 15,
    # tier_score deliberately absent
}
try:
    compute_tier_ml(MOCK_STATE["profile"], payload_missing_tier)
    print("FAILED: Agent 2 produced a prediction without Agent 1's tier -- this must not happen.")
except Agent1TierMissingError as e:
    print(f"PASSED: Agent 2 correctly refused -- {e}")

print("\n" + "=" * 70)
print("TEST 2 -- Agent 7 score is passed correctly (profile_strength_score=80)")
print("=" * 70)
raw, normalized = get_agent7_profile_strength({"extracurricular": {"profile_strength_score": 80}})
assert raw == 80.0 and abs(normalized - 0.80) < 1e-9, "Agent 7 score was not received/normalized correctly."
print(f"PASSED: raw={raw} (Agent 7's 0-100 scale) -> normalized={normalized} (Agent 2's 0-1 scale)")

print("\n" + "=" * 70)
print("TEST 3 -- Agent 7 score is not silently invented in integrated mode")
print("=" * 70)
state_no_agent7 = {**MOCK_STATE, "extracurricular": {}}
try:
    university_matching_agent(dict(state_no_agent7), standalone_mode=False)
    print("FAILED: Agent 2 proceeded without Agent 7's score in integrated mode.")
except Agent7ScoreMissingError as e:
    print(f"PASSED (integrated mode correctly refused): {e}")
# standalone_mode=True is the explicit, visible opt-out for isolated dev/testing:
_ = university_matching_agent(dict(state_no_agent7), standalone_mode=True)
print("PASSED (standalone_mode=True prints a visible DEV MODE warning and proceeds -- see above).")

print("\n" + "=" * 70)
print("TEST 4 -- Agent 7 score in the model FEATURE VECTOR (honest result: it is NOT, yet)")
print("=" * 70)
assert "extracurricular_score" not in FINAL_FEATURES, (
    "extracurricular_score unexpectedly in FINAL_FEATURES -- Section 6 found it has zero "
    "variance in the historical training data, so training on it would be fabricated signal."
)
print("CONFIRMED (expected): 'extracurricular_score' is NOT in FINAL_FEATURES -- the trained "
      "admission model does not use it (see Section 6/8). It DOES reach the matching-score "
      "ranking above (TEST 7's match_score differs with it) and the Groq explanation context.")

print("\n" + "=" * 70)
print("TEST 5 -- SHAP contains the feature (honest result: it does NOT, for the same reason)")
print("=" * 70)
sample_factors = result["matched_universities"][0]["programs"][0]["explainability"]
shap_feature_names = {f["feature"] for f in sample_factors}
assert "extracurricular_score" not in shap_feature_names
print(f"CONFIRMED (expected): SHAP factors {sorted(shap_feature_names)} do not include "
      f"'extracurricular_score', because it isn't a trained feature of admit_model. No SHAP "
      f"value is fabricated for it.")

print("\n" + "=" * 70)
print("TEST 6 -- different Agent 7 scores: does model behavior respond to the feature?")
print("=" * 70)
state_low = {**MOCK_STATE, "extracurricular": {"profile_strength_score": 20}}
state_high = {**MOCK_STATE, "extracurricular": {"profile_strength_score": 95}}
result_low = university_matching_agent(dict(state_low))
result_high = university_matching_agent(dict(state_high))
prog_low = result_low["matched_universities"][0]["programs"][0]
prog_high = result_high["matched_universities"][0]["programs"][0]
print(f"profile_strength_score=20 -> match_score={prog_low['match_score']}  "
      f"admit_probability={prog_low['admit_probability']}")
print(f"profile_strength_score=95 -> match_score={prog_high['match_score']}  "
      f"admit_probability={prog_high['admit_probability']}")
assert prog_low["match_score"] != prog_high["match_score"], (
    "match_score should move with Agent 7's score -- it is a real input to compute_final_score."
)
assert prog_low["admit_probability"] == prog_high["admit_probability"], (
    "admit_probability should NOT move -- extracurricular_score isn't a trained model feature "
    "yet; if this assertion fails, something started leaking it into the model unexpectedly."
)
print("PASSED (expected, honest behavior): match_score responds to Agent 7's score; "
      "admit_probability does not, because the trained model doesn't use it yet.")


TEST 7 (run first, others build on it) -- full existing flow still works
RAG, reranking, admission prediction, SHAP, Groq reasoning, ranking, DB persistence
POSTGRES_URL not set — skipping DB persistence (in-memory only).
Stanford University (Unknown, Unknown, Unknown) — overall_score=0.6880000233650208 admit_probability=0.511
  reasoning: Stanford’s MS Computer Science program is a Target match because the applicant’s GPA (3.44/4.0) positively interacts with the program’s high tier (gpa × tier = +0.0195) and the university’s strong tier score (86.4/100 contributes +0.0147), outweighing the modest negative applicant‑vs‑difficulty adjustment (‑0.0235). These combined model‑derived factors yield an overall match score of 0.803, indicating a strong alignment between the student’s academic profile and Stanford’s rigorous Computer Science curriculum.
    - MS Computer Science | agent1_tier=86.38 extracurricular_score=80.0 -> agent2_tier=Target match_score=0.8029999732971191 within_budget=Fa

In [31]:
print("=" * 70)
print("FINAL COUNSELING REPORT")
print("=" * 70)

profile = MOCK_STATE["profile"]
print(f"\nApplicant profile:")
print(f"  GPA: {profile['gpa']}/{profile['gpa_scale']} "
      f"(normalized to 4.0 scale: {normalize_gpa_to_4(profile['gpa'], profile['gpa_scale']):.2f})")
print(f"  GRE Quant: {profile['test_scores']['GRE']['quant']}")
print(f"  Target: {profile['target_degree']} in {profile['target_program']}")
print(f"  Agent 7 profile-strength score: {MOCK_STATE['extracurricular']['profile_strength_score']}/100")

for u in result["matched_universities"]:
    print(f"\n{'-'*70}")
    print(f"{u['university']}  ({u['type']}, {u['region']}, {u['country']})")
    print(f"  Overall match score: {u['overall_score']}")
    for p in u["programs"]:
        print(f"\n  Program: {p['program_name']}")
        print(f"    Agent 1 tier (university/program competitiveness, 0-100): {p['agent1_tier']}")
        print(f"    Agent 7 profile-strength score (0-100): {p['extracurricular_score']}")
        print(f"    Agent 2 admission probability: {p['admit_probability']:.1%}")
        print(f"    Classification: {p['tier']}  (Safe >= 65%, Target 35-65%, Reach < 35%)")
        print(f"    Top contributing factors (trained model features only):")
        for fac in p["explainability"]:
            direction = "raises" if fac["impact"] >= 0 else "lowers"
            print(f"      - {fac['feature']} {direction} the estimate (impact {fac['impact']:+.4f})")
        print(f"    Why this match: {p['explanation']}")

print(f"\n{'='*70}")
print("MODEL PERFORMANCE SUMMARY (for reference)")
print("=" * 70)
print(f"Production model: {PRODUCTION_MODEL_NAME} (RandomForestClassifier), tuned + calibrated "
      f"-- required by the architecture. CV-best in the research comparison was {best_model_name}.")
print(f"Validation-selected decision threshold: {best_threshold:.2f}")
for k, v in final_metrics.items():
    print(f"  Test {k}: {v:.4f}")
print(f"  Agent 1 Tier alone            -- test accuracy: {tier_only_acc:.4f}, ROC-AUC: {tier_only_auc:.4f}")
print(f"  Agent 1 Tier + Agent 2 (full) -- test accuracy: {final_metrics['Accuracy']:.4f}, "
      f"ROC-AUC: {final_metrics['ROC-AUC']:.4f}")

print(f"\n{'='*70}")
print("AGENT 7 INTEGRATION STATUS (honest summary)")
print("=" * 70)
print("- State contract: Agent 2 reads profile['tier_score'] (Agent 1, mandatory) and")
print("  extracurricular['profile_strength_score'] (Agent 7, 0-100, mandatory in integrated mode).")
print("- Agent 7's score DOES affect: the matching/ranking score (Section 5) and the Groq")
print("  explanation context (clearly labeled as background, not a model-derived factor).")
print("- Agent 7's score does NOT yet affect: the trained admission-probability model or its")
print("  SHAP output -- the historical admissions data has zero variance in this feature")
print("  (Section 6), so training on it would be fabricated signal, not a real ML contribution.")
print("- Closing this gap requires real historical profile_strength_score values for the")
print("  applicants in agent2's admits history -- not just Agent 7 running on new candidates.")


FINAL COUNSELING REPORT

Applicant profile:
  GPA: 8.6/10.0 (normalized to 4.0 scale: 3.44)
  GRE Quant: 165
  Target: MS in Computer Science
  Agent 7 profile-strength score: 80/100

----------------------------------------------------------------------
Stanford University  (Unknown, Unknown, Unknown)
  Overall match score: 0.6880000233650208

  Program: MS Computer Science
    Agent 1 tier (university/program competitiveness, 0-100): 86.38
    Agent 7 profile-strength score (0-100): 80.0
    Agent 2 admission probability: 51.1%
    Classification: Target  (Safe >= 65%, Target 35-65%, Reach < 35%)
    Top contributing factors (trained model features only):
      - applicant_vs_difficulty lowers the estimate (impact -0.0235)
      - gpa_x_tier raises the estimate (impact +0.0195)
      - tier_score raises the estimate (impact +0.0147)
    Why this match: Stanford’s MS Computer Science program is a Target match because the applicant’s GPA (3.44/4.0) positively interacts with the program